# K-Means avec `spark.ml` -- exemple sur jeu de donnees jouet

Ce notebook illustre l'ensemble du pipeline K-Means sur un jeu de donnees
synthetique de 120 stations de velos imaginaires, decrites par deux variables :

- `usage_matin` : taux d'occupation moyen entre 7 h et 10 h
- `usage_soir`  : taux d'occupation moyen entre 17 h et 20 h

Le jeu de donnees contient trois groupes naturels :

| Profil | usage_matin | usage_soir | Interpretation |
|--------|-------------|------------|----------------|
| Residentiel | eleve | faible | Les cyclistes *partent* le matin, les bornettes se vident |
| Bureau | faible | eleve | Les cyclistes *arrivent* le matin, les bornettes se remplissent |
| Mixte | modere | modere | Flux equilibre toute la journee |

L'objectif est de retrouver ces trois groupes automatiquement avec K-Means.


---
## 0. Initialisation de la SparkSession


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("KMeans-Jouet")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} pret.")


---
## 1. Creation du jeu de donnees jouet

Nous generons trois nuages de points gaussiens bien separes,
chacun representant un profil de station.


In [ ]:
import numpy as np
import pandas as pd

SEED = 42
rng  = np.random.default_rng(SEED)

def nuage(centre_matin, centre_soir, n, sigma=0.06):
    """Genere n stations autour d'un centre donne avec un bruit gaussien."""
    return pd.DataFrame({
        "usage_matin": np.clip(rng.normal(centre_matin, sigma, n), 0, 1),
        "usage_soir" : np.clip(rng.normal(centre_soir,  sigma, n), 0, 1),
    })

df_residentiel = nuage(centre_matin=0.80, centre_soir=0.20, n=40)
df_bureau      = nuage(centre_matin=0.20, centre_soir=0.80, n=40)
df_mixte       = nuage(centre_matin=0.50, centre_soir=0.50, n=40)

# Etiquettes reelles (on les cache a K-Means, elles serviront a valider)
df_residentiel["profil_reel"] = "residentiel"
df_bureau["profil_reel"]      = "bureau"
df_mixte["profil_reel"]       = "mixte"

df_pd = pd.concat([df_residentiel, df_bureau, df_mixte], ignore_index=True)
df_pd["station_id"] = range(1, len(df_pd) + 1)

print(f"{len(df_pd)} stations generees")
print(df_pd.groupby("profil_reel")[["usage_matin", "usage_soir"]].mean().round(3))


In [ ]:
# Conversion en DataFrame Spark
df = spark.createDataFrame(df_pd[["station_id", "usage_matin", "usage_soir"]])
df.printSchema()
df.show(8)


---
## 2. Exploration visuelle

Avant de lancer K-Means, on visualise les donnees brutes.
En pratique, cette etape n'est possible qu'en 2D ou 3D -- sur 24 dimensions
(profil horaire complet), on ne peut pas la faire directement.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

COULEURS = {"residentiel": "#e74c3c", "bureau": "#3498db", "mixte": "#2ecc71"}

fig, ax = plt.subplots(figsize=(6, 6))

for profil, groupe in df_pd.groupby("profil_reel"):
    ax.scatter(
        groupe["usage_matin"], groupe["usage_soir"],
        c=COULEURS[profil], alpha=0.7, s=50,
        label=profil.capitalize()
    )

ax.set_xlabel("Taux d'occupation moyen 7h-10h (usage_matin)")
ax.set_ylabel("Taux d'occupation moyen 17h-20h (usage_soir)")
ax.set_title("Jeu de donnees jouet -- profils reels (non vus par K-Means)")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("donnees_brutes.png", dpi=120, bbox_inches="tight")
plt.show()
print("Les trois groupes sont visuellement bien separes.")


---
## 3. Construction du Pipeline MLlib

Le pipeline se compose de trois etapes :

```
DataFrame  -->  VectorAssembler  -->  StandardScaler  -->  KMeans
               (2 colonnes          (centre et reduit     (affecte chaque
                -> 1 vecteur)        chaque feature)       point a un cluster)
```

### Etape 1 -- VectorAssembler

Reunit `usage_matin` et `usage_soir` en un seul vecteur colonne `features_brutes`.


In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline

# Etape 1 : assembler les deux features en un vecteur
assembler = VectorAssembler(
    inputCols=["usage_matin", "usage_soir"],
    outputCol="features_brutes"
)

# Verification intermediaire
df_assemble = assembler.transform(df)
df_assemble.select("station_id", "usage_matin", "usage_soir", "features_brutes").show(5)


### Etape 2 -- StandardScaler

Normalise les features : moyenne = 0, ecart-type = 1.

Sur ce jeu de donnees les deux variables sont deja dans [0, 1] et d'echelle
comparable -- le scaler ne changera pas beaucoup les resultats.
Mais en situation reelle (temperature en Celsius, vent en km/h, taux en [0,1]...),
les echelles tres differentes faussent les distances euclidiennes utilisees
par K-Means et rendent la normalisation indispensable.


In [ ]:
scaler = StandardScaler(
    inputCol="features_brutes",
    outputCol="features",
    withMean=True,    # soustrait la moyenne
    withStd=True      # divise par l'ecart-type
)

# Pour illustrer : fit sur les donnees, puis transform
scaler_model = scaler.fit(df_assemble)
print(f"Moyennes apprises : {scaler_model.mean}")
print(f"Ecarts-types      : {scaler_model.std}")


---
## 4. Methode du coude : choisir k

On entraine K-Means pour k = 2 a 7 et on releve l'**inertie** (WSSSE --
*Within Set Sum of Squared Errors*) a chaque fois.

L'inertie mesure la somme des carres des distances de chaque point a son
centroide. Elle diminue toujours quand k augmente -- le "coude" designe
le point ou le gain marginal devient negligeable.


In [ ]:
from pyspark.ml.evaluation import ClusteringEvaluator

resultats = []

for k in range(2, 8):
    kmeans_k = KMeans(
        featuresCol="features",
        predictionCol="cluster",
        k=k,
        seed=SEED,
        maxIter=50
    )
    pipeline_k = Pipeline(stages=[assembler, scaler, kmeans_k])
    model_k    = pipeline_k.fit(df)

    inertie    = model_k.stages[-1].summary.trainingCost
    df_pred_k  = model_k.transform(df)

    evaluator  = ClusteringEvaluator(
        featuresCol="features",
        predictionCol="cluster",
        metricName="silhouette"
    )
    silhouette = evaluator.evaluate(df_pred_k)

    resultats.append({"k": k, "inertie": round(inertie, 4),
                      "silhouette": round(silhouette, 4)})
    print(f"  k={k}  inertie={inertie:8.3f}  silhouette={silhouette:.4f}")

df_res = pd.DataFrame(resultats)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# -- Courbe du coude (inertie) --
ax1.plot(df_res["k"], df_res["inertie"], "o-", color="#2c3e50", linewidth=2)
ax1.axvline(x=3, color="#e74c3c", linestyle="--", alpha=0.7, label="k=3 (coude)")
ax1.set_xlabel("Nombre de clusters k")
ax1.set_ylabel("Inertie (WSSSE)")
ax1.set_title("Methode du coude")
ax1.legend(); ax1.grid(alpha=0.3)

# -- Score de silhouette --
ax2.bar(df_res["k"], df_res["silhouette"], color="#3498db", alpha=0.8)
ax2.axvline(x=3, color="#e74c3c", linestyle="--", alpha=0.7, label="k=3 (max)")
ax2.set_xlabel("Nombre de clusters k")
ax2.set_ylabel("Score de silhouette")
ax2.set_title("Score de silhouette (plus haut = meilleur)")
ax2.legend(); ax2.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("methode_coude.png", dpi=120, bbox_inches="tight")
plt.show()

print()
print(df_res.to_string(index=False))
print()
print("-> k=3 presente le coude le plus net et le score de silhouette le plus eleve.")
print("   C'est coherent avec la structure reelle du jeu de donnees.")


---
## 5. Entrainement final avec k = 3


In [ ]:
K = 3

kmeans = KMeans(
    featuresCol="features",
    predictionCol="cluster",
    k=K,
    maxIter=100,
    seed=SEED
)

pipeline = Pipeline(stages=[assembler, scaler, kmeans])
model    = pipeline.fit(df)

print("Pipeline entraine.")
print(f"Stages : {[type(s).__name__ for s in model.stages]}")


In [ ]:
# Application du modele : chaque station recoit un numero de cluster (0, 1 ou 2)
df_pred = model.transform(df)
df_pred.select("station_id", "usage_matin", "usage_soir", "features", "cluster").show(8)


In [ ]:
# Centroides appris (dans l'espace des features normalisees)
kmeans_model = model.stages[-1]

print(f"Inertie finale      : {kmeans_model.summary.trainingCost:.4f}")
print(f"Taille des clusters : {kmeans_model.summary.clusterSizes}")
print()

# Pour interpreter les centroides, on les exprime dans l'espace original
# en appliquant la transformation inverse du scaler
scaler_model_final = model.stages[1]
mean_ = scaler_model_final.mean.toArray()
std_  = scaler_model_final.std.toArray()

print("Centroides (espace original) :")
print(f"  {'Cluster':<10} {'usage_matin':>14} {'usage_soir':>14}")
print("  " + "-" * 40)
for i, c in enumerate(kmeans_model.clusterCenters()):
    c_original = c * std_ + mean_   # transformation inverse : x = z*std + mean
    print(f"  {i:<10} {c_original[0]:>14.3f} {c_original[1]:>14.3f}")


---
## 6. Visualisation des resultats

On compare les clusters trouves par K-Means (gauche) avec les vrais profils (droite).


In [ ]:
# Recuperation des predictions en Pandas pour la visualisation
df_result = (
    df_pred
    .select("station_id", "usage_matin", "usage_soir", "cluster")
    .toPandas()
    .merge(df_pd[["station_id", "profil_reel"]], on="station_id")
)

# Correspondance cluster -> couleur (on assigne une couleur par cluster detecte)
COULEURS_CLUSTER = {0: "#e67e22", 1: "#9b59b6", 2: "#1abc9c"}

# Centroides dans l'espace original (calcules ci-dessus)
centroides_orig = [
    c * std_ + mean_ for c in kmeans_model.clusterCenters()
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))

# -- Gauche : clusters K-Means --
for cluster_id, groupe in df_result.groupby("cluster"):
    ax1.scatter(
        groupe["usage_matin"], groupe["usage_soir"],
        c=COULEURS_CLUSTER[cluster_id],
        alpha=0.75, s=55, label=f"Cluster {cluster_id}"
    )

for i, c in enumerate(centroides_orig):
    ax1.scatter(c[0], c[1],
                c=COULEURS_CLUSTER[i], s=250, marker="*",
                edgecolors="black", linewidths=1.2, zorder=5)
    ax1.annotate(f" C{i}", (c[0], c[1]), fontsize=10, fontweight="bold")

ax1.set_xlabel("usage_matin"); ax1.set_ylabel("usage_soir")
ax1.set_title("Clusters trouves par K-Means (k=3)")
ax1.set_xlim(0, 1); ax1.set_ylim(0, 1)
ax1.legend(title="Cluster detecte"); ax1.grid(alpha=0.3)

# -- Droite : vrais profils --
for profil, groupe in df_result.groupby("profil_reel"):
    ax2.scatter(
        groupe["usage_matin"], groupe["usage_soir"],
        c=COULEURS[profil], alpha=0.75, s=55, label=profil.capitalize()
    )

ax2.set_xlabel("usage_matin"); ax2.set_ylabel("usage_soir")
ax2.set_title("Vrais profils (non utilises par K-Means)")
ax2.set_xlim(0, 1); ax2.set_ylim(0, 1)
ax2.legend(title="Profil reel"); ax2.grid(alpha=0.3)

plt.suptitle("K-Means : comparaison clusters detectes vs profils reels",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("kmeans_resultats.png", dpi=120, bbox_inches="tight")
plt.show()


---
## 7. Evaluation de la qualite du clustering

### Mesure interne : score de silhouette

Le score de silhouette d'un point mesure a quel point il est proche des autres
points de son cluster (cohesion) par rapport aux points du cluster voisin
(separation).

```
s(i) = (b(i) - a(i)) / max(a(i), b(i))

a(i) = distance moyenne aux autres points du MEME cluster
b(i) = distance moyenne aux points du cluster le PLUS PROCHE

s(i) proche de  1 : bien place dans son cluster
s(i) proche de  0 : sur la frontiere entre deux clusters
s(i) proche de -1 : probablement mal assigne
```


In [ ]:
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator_final = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="cluster",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)
score_final = evaluator_final.evaluate(df_pred)
print(f"Score de silhouette (k=3) : {score_final:.4f}")
print()
print("Interpretation :")
if score_final > 0.70:
    print("  > 0.70 : structure tres forte, clusters bien separes.")
elif score_final > 0.50:
    print("  > 0.50 : structure raisonnable.")
elif score_final > 0.25:
    print("  > 0.25 : structure faible, chevauchement possible.")
else:
    print("  < 0.25 : pas de structure claire.")


In [ ]:
# Mesure externe : comparaison avec les vrais profils
# (uniquement possible quand on connait les vraies etiquettes -- cas jouet)
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# Encodage des labels reels en entiers pour sklearn
profil_to_int = {"residentiel": 0, "bureau": 1, "mixte": 2}
labels_reels   = df_result["profil_reel"].map(profil_to_int).values
labels_kmeans  = df_result["cluster"].values

ari  = adjusted_rand_score(labels_reels, labels_kmeans)
nmi  = normalized_mutual_info_score(labels_reels, labels_kmeans)

print(f"Adjusted Rand Index (ARI)              : {ari:.4f}")
print(f"Normalized Mutual Information (NMI)    : {nmi:.4f}")
print()
print("Interpretation :")
print("  ARI=1 : clustering parfait | ARI=0 : aleatoire | ARI<0 : pire qu'aleatoire")
print("  NMI=1 : information parfaite | NMI=0 : independance totale")
print()
print("Note : ARI et NMI ne sont disponibles qu'avec des etiquettes reelles.")
print("En production (pas d'etiquettes), on se limite au score de silhouette.")


---
## 8. Matrice de correspondance clusters / profils reels

K-Means ne nomme pas ses clusters -- il leur donne des numeros arbitraires
(0, 1, 2) qui peuvent varier d'une execution a l'autre.
On construit une matrice de correspondance pour savoir quel numero
correspond a quel profil reel.


In [ ]:
correspondance = (
    df_result
    .groupby(["cluster", "profil_reel"])
    .size()
    .unstack(fill_value=0)
)
print("Matrice de correspondance (lignes = cluster K-Means, colonnes = profil reel) :")
print(correspondance)
print()
print("Chaque cluster detecte correspond quasi-parfaitement a un profil reel.")
print("Les quelques points mal classes sont ceux situes a la frontiere des nuages.")

n_bien_classes = correspondance.values.max(axis=1).sum()
print(f"\nPrecision globale : {n_bien_classes}/{len(df_result)} = {n_bien_classes/len(df_result):.1%}")


---
## 9. Sauvegarde et rechargement du Pipeline

Un `PipelineModel` entraine peut etre sauvegarde sur disque et recharge
pour predire sur de nouvelles donnees -- sans avoir a reentrain er.


In [ ]:
from pyspark.ml import PipelineModel

# Sauvegarde (cree un repertoire, pas un fichier unique)
chemin = "/tmp/kmeans_jouet_pipeline"
model.write().overwrite().save(chemin)
print(f"Modele sauvegarde dans : {chemin}")

import os
for racine, dossiers, fichiers in os.walk(chemin):
    niveau = racine.replace(chemin, "").count(os.sep)
    indent = "  " * niveau
    print(f"{indent}{os.path.basename(racine)}/")
    for f in fichiers[:3]:   # on limite l'affichage
        print(f"{indent}  {f}")


In [ ]:
# Rechargement
model_recharge = PipelineModel.load(chemin)
print(f"Stages recharges : {[type(s).__name__ for s in model_recharge.stages]}")
print()

# Application sur de nouvelles stations (jamais vues)
nouvelles_stations = spark.createDataFrame([
    (201, 0.82, 0.18),   # clairement residentiel
    (202, 0.19, 0.83),   # clairement bureau
    (203, 0.51, 0.49),   # clairement mixte
    (204, 0.65, 0.35),   # ambigu ?
], ["station_id", "usage_matin", "usage_soir"])

df_nouvelles_pred = model_recharge.transform(nouvelles_stations)
df_nouvelles_pred.select("station_id", "usage_matin", "usage_soir", "cluster").show()


---
## Bilan

### Ce que cet exemple illustre

| Etape | Classe MLlib | Role |
|-------|-------------|------|
| Assemblage | `VectorAssembler` | Reunir n colonnes en un vecteur |
| Normalisation | `StandardScaler` | Mettre les features a la meme echelle |
| Clustering | `KMeans` | Affecter chaque point a un cluster |
| Orchestration | `Pipeline` | Enchaîner les etapes proprement |
| Evaluation interne | `ClusteringEvaluator` | Score de silhouette |
| Persistance | `PipelineModel` | Sauvegarder et recharger |

### Ce que K-Means ne sait pas faire

- Il suppose des clusters **convexes et de taille comparable**.
  Des clusters en forme de croissant ou de densite tres differente
  le mettent en echec.
- Il est **sensible aux outliers** : un point tres eloigne peut tirer
  un centroide loin de la masse principale.
- Il ne peut pas determiner k automatiquement : la methode du coude
  (et le score de silhouette) restent des heuristiques.

### Passage au projet ClimaCity

Sur le projet, le vecteur de features est de dimension 24 (un taux moyen
par heure de la journee) et non 2. La visualisation directe est impossible,
mais la procedure est exactement la meme. Le score de silhouette et la methode
du coude restent les outils de reference pour choisir k.


In [ ]:
spark.stop()
print("SparkSession arretee.")
